In [ ]:
#@title Parameters — edit these fields

#@markdown ### 出力設定
#@markdown **出力ファイル名** — ダウンロードされる.docxファイルの名前
file_name_download = 'kawaguchi_pioneering.docx' #@param {type:"string"}

#@markdown ---
#@markdown ### 期間設定
#@markdown **開始日** — この日付より後の業績を収集 (YYYY-MM-DD)
globalmindate='2018-04-01' #@param {type:"string"}
#@markdown **終了日** — この日付より前の業績を収集 (YYYY-MM-DD)
globalmaxdate='2026-12-31' #@param {type:"string"}

#@markdown ---
#@markdown ### 書式設定
#@markdown **論文ナンバリング** — Trueで連番付き
numberingPapers = True #@param {type:"boolean"}
#@markdown **査読ありのみ** — Trueで査読済み論文のみ
peer_reviewed = False #@param {type:"boolean"}
#@markdown **DOIリンクを付与**
include_doi = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 研究者情報
#@markdown **researchmap ID** — 対象研究者のresearchmap ID
researchmap_id = 'kyogok' #@param {type:"string"}
#@markdown **見出しに使う日本語名** — セクションタイトルの接頭辞
display_name_jp = '川口喬吾' #@param {type:"string"}

In [ ]:
#@title Imports and setup
import requests, json, sys, os, time, re
import numpy as np
if 'google.colab' in str(get_ipython()):
    %pip install python-docx
    from google.colab import files, auth
    outputdirectory = ''
else:
    outputdirectory = '../docx-researchmap-outputs/'
    os.makedirs(outputdirectory, exist_ok=True)
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_UNDERLINE
file_name = outputdirectory + file_name_download

In [ ]:
#@title Fetch researcher profile
url = "https://api.researchmap.jp/"
profile = requests.get(url + researchmap_id).json()
if 'error' in profile:
    raise RuntimeError(f"Could not fetch profile for '{researchmap_id}': {profile['error']}")

surname_en = profile.get('family_name', {}).get('en', '')
given_en   = profile.get('given_name',  {}).get('en', '')
surname_ja = profile.get('family_name', {}).get('ja', '')
given_ja   = profile.get('given_name',  {}).get('ja', '')
fullname   = (given_en + ' ' + surname_en).strip()
fullnameJP = (surname_ja + ' ' + given_ja).strip()
print(f"Researcher: {researchmap_id}  —  {fullname} / {fullnameJP}")

In [ ]:
#@title Parse researcher data
allnames         = [fullname]
allSurname       = [surname_en]
allnamesJP       = [fullnameJP]
allmembers       = [researchmap_id]
allDaihyoBuntan  = ['D']
allmindate       = [globalmindate]
allmaxdate       = [globalmaxdate]

nameList         = [fullname]
daihyobuntanList = ['D']

In [ ]:
#@title Helper functions
def SurnameLast(namesDic, sn):
    """Return list of names with surname moved to the end (Western order)."""
    oldnamelist = [indiv['name'].replace(',', '').replace('.', '') for indiv in namesDic]
    swap = 0
    for name in oldnamelist:
        if sn in name.split(' '):
            swap = 1 if name.split(' ').index(sn) == 0 else 0
            break
    if not swap:
        return oldnamelist
    return [' '.join(name.split(' ')[1:] + [name.split(' ')[0]]) for name in oldnamelist]


def to_surname_initial(name):
    """Convert 'Firstname Middlename Surname' → 'Surname F'.
    Japanese names (no ASCII letters) are returned unchanged."""
    if not name:
        return name
    if not re.search(r'[A-Za-z]', name):
        return name
    tokens = [t for t in name.split(' ') if t]
    if len(tokens) == 1:
        return tokens[0]
    surname = tokens[-1]
    initials = ''.join(t[0].upper() for t in tokens[:-1] if t and t[0].isalpha())
    return f"{surname} {initials}" if initials else surname


def ReturnDictWOerror(d, key, nodata):
    return d[key] if key in d.keys() else nodata

def ReturnDictContent(d, key, key1, nodata=''):
    v  = ReturnDictWOerror(d, key,  nodata)
    v1 = ReturnDictWOerror(d, key1, nodata)
    return v if v != nodata else v1

def commaR(vol, spage):
    if vol == '' and spage == '':
        return ''
    if vol == '' or spage == '':
        return ' '
    return ', '

def strip_html_tags(text):
    return re.sub(r'<[^>]+>', '', text)

BIORXIV_DOI_PREFIXES = ('10.1101/', '10.64898/')

_ACRONYM_FIXES = {
    'Bmc': 'BMC', 'Acs': 'ACS', 'Rsc': 'RSC', 'Ieee': 'IEEE',
    'Plos': 'PLoS', 'Embo': 'EMBO', 'Febs': 'FEBS', 'Faseb': 'FASEB',
    'Pnas': 'PNAS', 'Dna': 'DNA', 'Rna': 'RNA', 'Mrna': 'mRNA',
    'Iscience': 'iScience', 'Elife': 'eLife', 'Eneuro': 'eNeuro',
    'Biorxiv': 'bioRxiv', 'Medrxiv': 'medRxiv', 'Chemrxiv': 'ChemRxiv',
    'Prx': 'PRX', 'Jacs': 'JACS', 'Aiche': 'AIChE', 'Jsme': 'JSME', 'Jsap': 'JSAP',
}

def _fix_acronyms(text):
    for wrong, right in _ACRONYM_FIXES.items():
        text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
    return text

_jname_cache = {}
def _strip_abbrev_periods(s):
    """Remove ISO-4 trailing periods from abbreviated tokens (e.g. 'Mol. Cell' -> 'Mol Cell').
    Collapse repeated whitespace afterwards."""
    return re.sub(r'\s+', ' ', s.replace('.', '')).strip()

def abbreviate_journal(name):
    if not name:
        return name
    if name in _jname_cache:
        return _jname_cache[name]
    query_name = name.title()
    try:
        r = requests.get('https://abbreviso.toolforge.org/abbreviso/a/' + requests.utils.quote(query_name), timeout=5)
        if r.status_code == 200:
            abbrev = _strip_abbrev_periods(_fix_acronyms(r.text.strip()))
            if abbrev:
                _jname_cache[name] = abbrev
                return abbrev
    except Exception:
        pass
    fallback = _strip_abbrev_periods(_fix_acronyms(query_name))
    _jname_cache[name] = fallback
    return fallback

def add_underlined_run(paragraph, name, nameList, daihyobuntanList):
    """Add a run with D=double-underline, B=single-underline, else plain."""
    if name in nameList:
        role = daihyobuntanList[nameList.index(name)]
        if role == 'D':
            paragraph.add_run(name).underline = WD_UNDERLINE.DOUBLE
        elif role == 'B':
            paragraph.add_run(name).underline = True
        else:
            paragraph.add_run(name)
    else:
        paragraph.add_run(name)

def sort_by_date_desc(items_dict):
    keys = list(items_dict.keys())
    datelist = [items_dict[k]['date'] for k in keys]
    arg = np.argsort(datelist)[::-1]
    return keys, arg

In [ ]:
#@title Download from researchmap API
url = "https://api.researchmap.jp/"
itemslist = ["published_papers", "presentations"]
jsonfiles = {}
for name in allmembers:
    print('downloading: ' + name)
    jsonfiles[name] = {}
    for it in itemslist:
        r1 = requests.get(url + name + '/' + it)
        jsonfiles[name][it] = json.loads(r1.text)
        if 'error' in jsonfiles[name][it].keys():
            print(jsonfiles[name][it]['error'])
            print("  error in: " + it)

In [ ]:
#@title Build papers dictionary
PapersDict = {}
doilist, doiDict = [], {}
titlelist, titleDict = [], {}
i = 0
for ids, fullname, surname, dh, mindate, maxdate in zip(allmembers, allnames, allSurname, allDaihyoBuntan, allmindate, allmaxdate):
    dfP = jsonfiles[ids]["published_papers"]
    if 'items' not in dfP.keys():
        continue
    for dfs in dfP['items']:
        if "authors" not in dfs.keys():
            continue
        if not ('identifiers' in dfs.keys() and dfs["publication_date"] >= mindate and dfs["publication_date"] <= maxdate):
            continue
        doinum = [0]
        if 'doi' in dfs['identifiers'].keys():
            doinum = dfs['identifiers']['doi']

        entry = {'issues': False, 'preprint': False}
        correspo = is_first = False
        roles = dfs.get("published_paper_owner_roles", []) or []
        if "corresponding" in roles:
            correspo = True
        if "first" in roles:
            is_first = True

        jname = ''
        if "publication_name" in dfs.keys():
            jname = strip_html_tags(ReturnDictContent(dfs["publication_name"], 'en', 'ja', ''))

        if jname.upper() == 'ARXIV':
            entry['preprint'] = True
            jname = dfs['identifiers']['arxiv_id'][0] + ' (preprint)' if 'arxiv_id' in dfs['identifiers'] else 'arXiv'
        if jname.upper() == 'BIORXIV':
            jname = 'bioRxiv'
            entry['preprint'] = True
        if "publication_name" not in dfs.keys():
            if 'arxiv_id' in dfs['identifiers']:
                jname = dfs['identifiers']['arxiv_id'][0] + ' (preprint)'
                entry['preprint'] = True
            elif doinum[0] != 0:
                jname = 'DOI: ' + doinum[0]
                entry['preprint'] = True
            else:
                jname = 'journal unspecified'
                entry['issues'] = True

        if doinum[0] != 0 and str(doinum[0]).startswith(BIORXIV_DOI_PREFIXES):
            if jname == '' or jname == 'DOI: ' + doinum[0]:
                jname = 'bioRxiv'
                entry['preprint'] = True

        if not entry['preprint'] and not entry['issues']:
            jname = abbreviate_journal(jname)

        # Full-name Western order, then convert to 'Surname F' initials
        Sname_full = SurnameLast(ReturnDictContent(dfs["authors"], 'en', 'ja', ''), surname)
        Sname = [to_surname_initial(n) for n in Sname_full]
        # Identify which index belongs to our target researcher (full name match)
        target_idx = Sname_full.index(fullname) if fullname in Sname_full else -1

        spage = dfs.get("starting_page", '') or ''
        epage = dfs.get("ending_page", '') or ''
        page_range = spage if not epage else (spage + '-' + epage if spage else '')
        vol = dfs.get("volume", '') or ''

        papertitle = strip_html_tags(ReturnDictContent(dfs['paper_title'], 'en', 'ja', ''))
        papid = papertitle.upper().rstrip('.')

        if doinum in doilist:
            doiDict[doinum[0]]['name'] += [fullname]
            doiDict[doinum[0]]['Corresp'] += [correspo]
        else:
            doiDict[doinum[0]] = {'name': [fullname], 'Corresp': [correspo], 'count': 0}
            doilist.append(doinum[0])
        if papid in titlelist:
            titleDict[papid]['name'] += [fullname]
            titleDict[papid]['Corresp'] += [correspo]
        else:
            titlelist.append(papid)
            titleDict[papid] = {'name': [fullname], 'Corresp': [correspo], 'count': 0}

        PapersDict[i] = entry
        PapersDict[i].update({
            'papid': papid,
            'researcher': fullname,
            'authors_full': Sname_full,
            'authors': Sname,
            'target_idx': target_idx,
            'title': papertitle,
            'year': dfs["publication_date"][:4],
            'journal': jname,
            'volume': vol,
            'pages': page_range,
            'date': dfs["publication_date"],
            'referee': ReturnDictContent(dfs, 'referee', 'referee', False),
            'doi': doinum[0],
            'Daihyo': dh,
            'Corresp': correspo,
            'IsFirst': is_first,
        })
        i += 1

In [ ]:
#@title Build talks dictionary (invited only; split international/domestic downstream)
TalksDict = {}
i = 0
for ids, fullname, fullnameJP, dh, mindate, maxdate in zip(allmembers, allnames, allnamesJP, allDaihyoBuntan, allmindate, allmaxdate):
    dfPr = jsonfiles[ids]["presentations"]
    if 'items' not in dfPr.keys():
        continue
    for dfs in dfPr['items']:
        needed = ["presentation_title", "event", 'publication_date', "presenters"]
        if not all(a in dfs.keys() for a in needed):
            continue
        if not (dfs["publication_date"] >= mindate and dfs["publication_date"] <= maxdate):
            continue
        if not dfs.get('invited', False):
            continue
        intl = bool(dfs.get('is_international_presentation', False))
        # English title if available (international), else Japanese (domestic)
        if intl:
            ename = strip_html_tags(ReturnDictContent(dfs["event"], 'en', 'ja', ''))
            ptitle = strip_html_tags(ReturnDictContent(dfs["presentation_title"], 'en', 'ja', ''))
        else:
            ename = strip_html_tags(ReturnDictContent(dfs["event"], 'ja', 'en', ''))
            ptitle = strip_html_tags(ReturnDictContent(dfs["presentation_title"], 'ja', 'en', ''))
        TalksDict[i] = {
            'presenter': fullname,
            'event': ename,
            'presentation_title': ptitle,
            'date': dfs["publication_date"],
            'international': intl,
        }
        i += 1

In [ ]:
#@title Generate docx
from datetime import datetime

# ---- Filter papers ----
if peer_reviewed:
    PapersDictSelected = {k: v for k, v in PapersDict.items()
                          if v['date'] > globalmindate and v['date'] < globalmaxdate and v['referee']}
else:
    PapersDictSelected = {k: v for k, v in PapersDict.items()
                          if v['date'] > globalmindate and v['date'] < globalmaxdate}

keys, arg = sort_by_date_desc(PapersDictSelected)

document = Document()

# ---- 主要論文 ----
document.add_paragraph(display_name_jp + '主要論文')
inds = 0
for r in arg:
    pap = PapersDictSelected[keys[r]]
    if pap['issues']:
        document.add_paragraph('***')
    inds += 1
    prefix = (str(inds) + '.\t') if numberingPapers else ''
    p = document.add_paragraph(prefix)

    # --- Author list with 'and' before the last author and target-researcher markers ---
    authors = pap['authors']
    authors_full = pap['authors_full']
    target_idx = pap['target_idx']
    n_auth = len(authors)
    for idx, (nm_init, nm_full) in enumerate(zip(authors, authors_full)):
        # separator before this author
        if idx > 0:
            if idx == n_auth - 1 and n_auth >= 2:
                p.add_run(', and ' if n_auth > 2 else ' and ')
            else:
                p.add_run(', ')
        # the name itself — target researcher gets underline; others plain
        if idx == target_idx:
            run = p.add_run(nm_init)
            role = daihyobuntanList[0] if daihyobuntanList else ''
            if role == 'D':
                run.underline = WD_UNDERLINE.DOUBLE
            elif role == 'B':
                run.underline = True
            # role markers after the target researcher's name
            marks = ''
            if pap['IsFirst']:
                marks += '*'
            if pap['Corresp']:
                marks += '†'
            if marks:
                p.add_run(marks)
        else:
            p.add_run(nm_init)
    p.add_run('. ')

    # --- (Year) Title. Journal vol, pp. doi_link ---
    p.add_run('(' + pap['year'] + ') ')
    title_clean = pap['title'].rstrip('.').strip()
    p.add_run(title_clean + '. ')
    jpart = pap['journal']
    if pap['volume']:
        jpart += ' ' + pap['volume']
    if pap['pages']:
        jpart += (', ' if pap['volume'] else ' ') + pap['pages']
    p.add_run(jpart + '.')
    if include_doi and pap['doi'] and pap['doi'] != 0:
        p.add_run(' doi.org/' + str(pap['doi']))

# ---- Talks: split international vs domestic ----
TalksDictSelected = {k: v for k, v in TalksDict.items()
                     if v['date'] > globalmindate and v['date'] < globalmaxdate}
intl_talks = {k: v for k, v in TalksDictSelected.items() if v['international']}
dom_talks  = {k: v for k, v in TalksDictSelected.items() if not v['international']}

def _fmt_date_intl(d):
    # Preserve the ISO date; users can hand-edit to 'Mon D, YYYY' if desired
    return d

def _fmt_date_jp(d):
    # '2025-09-04' -> '2025/9/4'
    try:
        y, m, day = d.split('-')
        return f"{y}/{int(m)}/{int(day)}"
    except Exception:
        return d

# --- 国際招待講演 ---
document.add_paragraph('')
document.add_paragraph('国際招待講演')
if intl_talks:
    keys, arg = sort_by_date_desc(intl_talks)
    inds = 0
    for r in arg:
        inds += 1
        t = intl_talks[keys[r]]
        p = document.add_paragraph(str(inds) + '.\t')
        p.add_run((t['event'].rstrip('.') + '. ') if t['event'] else '')
        p.add_run((t['presentation_title'].rstrip('.') + '. ') if t['presentation_title'] else '')
        p.add_run(_fmt_date_intl(t['date']))

# --- 国内招待講演 ---
document.add_paragraph('')
document.add_paragraph('国内招待講演')
if dom_talks:
    keys, arg = sort_by_date_desc(dom_talks)
    inds = 0
    for r in arg:
        inds += 1
        t = dom_talks[keys[r]]
        p = document.add_paragraph(str(inds) + '.\t')
        p.add_run(t['event'] if t['event'] else '')
        if t['presentation_title']:
            p.add_run(" \u2018" + t['presentation_title'] + "\u2019")
        p.add_run('\u3001' + _fmt_date_jp(t['date']))

# ---- Save ----
try:
    document.save(file_name)
except PermissionError:
    stem, ext = os.path.splitext(file_name)
    file_name = stem + '_' + datetime.now().strftime('%Y%m%d_%H%M%S') + ext
    document.save(file_name)
    print('Original file was locked (open in another app?). Saved as: ' + file_name)
print('Saved:', file_name)

In [ ]:
#@title Download file
if 'google.colab' in str(get_ipython()):
    files.download(file_name)